In [ ]:
import os, sys, json, shutil, numpy as np, subprocess, time
from pathlib import Path

COMP = Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
WORK = Path('/kaggle/working')
sys.path.insert(0, str(COMP / 'kickmix'))
sys.path.insert(0, str(COMP / 'verifiers'))

SRC = Path('/kaggle/input/datasets/michaelrustad/vqcc-dfthc-r10-merged-groups/src')
sys.path.insert(0, str(SRC))

import types
dfthcsos = types.ModuleType('dfthcsos')
sys.modules['dfthcsos'] = dfthcsos
scoreinfra = types.ModuleType('dfthcsos.scoreinfra')
sys.modules['dfthcsos.scoreinfra'] = scoreinfra
dfthcsos.scoreinfra = scoreinfra
scoreinfra.__path__ = []
import sys
import os
print('Input dirs:', os.listdir('/kaggle/input'))
qv_path = '/kaggle/input/datasets/evanpieser/qcompile-verifiable'
print('qcompile-verifiable exists:', os.path.exists(qv_path))
if os.path.exists(qv_path):
    print('Contents:', os.listdir(qv_path))
    sk = os.path.join(qv_path, 'starter_kit')
    if os.path.exists(sk):
        print('starter_kit contents:', os.listdir(sk)[:10])
        sys.path.insert(0, sk)
        starter = os.path.join(sk, 'starter')
        if os.path.exists(starter):
            print('starter contents:', os.listdir(starter)[:10])
            sys.path.insert(0, starter)
        else:
            print('starter NOT found')
    else:
        print('starter_kit NOT found')
else:
    print('qcompile-verifiable NOT found')
sys.path.insert(0, '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')
import dfthc_math as dm
for attr in dir(dm):
    if not attr.startswith('__'):
        setattr(scoreinfra, attr, getattr(dm, attr))
sys.path.insert(0, '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')
import circuits as vc
circuits_mod = types.ModuleType('dfthcsos.scoreinfra.circuits')
for attr in dir(vc):
    if not attr.startswith('__'):
        setattr(circuits_mod, attr, getattr(vc, attr))
sys.modules['dfthcsos.scoreinfra.circuits'] = circuits_mod
scoreinfra.circuits = circuits_mod

bp_members = Path('/kaggle/input/datasets/busyaprime/33-qcc-v8-members')
sub_dir = WORK / 'submission'
sub_dir.mkdir(exist_ok=True)
(sub_dir / 'dfthcsos').mkdir(exist_ok=True)

shutil.copy2(bp_members / 'invsqrt.kmb', sub_dir / 'invsqrt.kmb')
shutil.copy2(bp_members / 'dqi.kmb', sub_dir / 'dqi.kmb')
for p in bp_members.glob('dfthcsos__*.kmb'):
    shutil.copy2(p, sub_dir / 'dfthcsos' / p.name.replace('dfthcsos__', ''))

from starter.task1_factorization.solution_io import read_factorization
from starter.task2_angles.encode import TREES, encode_inputs
import glob
import glob
print('Searching for starter module...')
print('Input dirs:', os.listdir('/kaggle/input'))
for d in os.listdir('/kaggle/input'):
    print(' ', d)
    full_path = os.path.join('/kaggle/input', d)
    try:
        print('   ', os.listdir(full_path)[:10])
    except:
        pass
    # Check if this is the competition
    if d == 'competitions':
        comp_path = os.path.join(full_path, 'google-verifiable-quantum-circuit-compilation')
        if os.path.exists(comp_path):
            print('   Competition found at:', comp_path)
            sys.path.insert(0, comp_path)
            sys.path.insert(0, os.path.join(comp_path, 'kickmix'))
            sys.path.insert(0, os.path.join(comp_path, 'verifiers'))
    # Check if this is the qcompile dataset
    for sub in ['starter_kit', 'qcompile']:
        sk = os.path.join(full_path, sub)
        if os.path.exists(sk):
            print(f'   {sub} found at:', sk)
            if sk not in sys.path:
                sys.path.insert(0, sk)
    # Try to find starter in subdirectories
    for root, dirs, files in os.walk(full_path):
        if 'solution_io.py' in files:
            print('   FOUND solution_io at:', root)
            sp = os.path.dirname(os.path.dirname(os.path.dirname(root)))
            if sp not in sys.path:
                sys.path.insert(0, sp)
            break
from verifier.score import load_inputs

bp_npz = bp_members / 'dfthcsos__submission.npz'
params, _ = read_factorization(bp_npz)
inputs = load_inputs('FeMoCo76o113e', params)
npz = np.load(bp_npz, allow_pickle=False)
tree_edges = np.array(npz['tree_edges'])
words = encode_inputs(inputs, tree_edges, 8, 'nearest')

data = {k: np.array(npz[k]) for k in npz.keys()}
data['brot'] = np.int64(8)
data['angle_words'] = words
data['words1'] = words[:76]
data['words2'] = words[76:]

enriched = WORK / 'enriched.npz'
np.savez(enriched, **data)
shutil.copy2(enriched, sub_dir / 'dfthcsos' / 'submission.npz')

print('Setup complete')


In [ ]:
IN = sub_dir / 'dfthcsos' / 'submission.npz'
RUNS = WORK / 'runs'
RUNS.mkdir(exist_ok=True)

print('Step 1: finish.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'finish.py'), str(IN), str(RUNS / 'fin.npz'), '--kappa', '0'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC) + os.pathsep + str(COMP / 'kickmix') + os.pathsep + str(COMP / 'verifiers') + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit' + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')
print('Output exists:', (RUNS / 'fin.npz').exists())


In [ ]:
print('Step 2: gauge.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'gauge.py'), str(RUNS / 'fin.npz'), str(RUNS / 'g.npz'), '--sweeps', '60'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC) + os.pathsep + str(COMP / 'kickmix') + os.pathsep + str(COMP / 'verifiers') + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit' + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')
print('Output exists:', (RUNS / 'g.npz').exists())


In [ ]:
# Save the optimized factorization
import shutil
final = RUNS / 'g.npz'
if not final.exists():
    final = RUNS / 'fin.npz'
print('Using:', final)
shutil.copy2(final, WORK / 'optimized.npz')
print('Saved optimized.npz')

# Also copy to submission dir
shutil.copy2(final, sub_dir / 'dfthcsos' / 'submission.npz')
print('Done')
